# Notebook 14: Introduction to Quantum Machine Learning

**Series: Quantum Computing from Ground Up**

---

## Learning Objectives

1. Understand the landscape of Quantum Machine Learning (QML)
2. Master data encoding strategies: basis, amplitude, and angle encoding
3. Build parameterized quantum circuits (PQCs)
4. Learn about quantum feature maps and kernels
5. Understand the barren plateaus problem
6. Implement a quantum classifier using PennyLane

## 1. What is Quantum Machine Learning?

### 1.1 The QML Landscape

Quantum Machine Learning sits at the intersection of quantum computing and machine learning. The field can be organized into four categories:

| | Classical Data | Quantum Data |
|---|---|---|
| **Classical Processing** | Classical ML | Quantum state tomography |
| **Quantum Processing** | QML (this notebook) | Quantum-quantum learning |

We focus on **using quantum computers to process classical data** — the most active area of current research.

### 1.2 The QML Pipeline

A typical QML algorithm follows these steps:

1. **Data Encoding:** Map classical data $\mathbf{x} \in \mathbb{R}^d$ to quantum states $|\phi(\mathbf{x})\rangle$
2. **Quantum Processing:** Apply parameterized quantum circuit $U(\boldsymbol{\theta})$
3. **Measurement:** Extract classical information via expectation values
4. **Classical Optimization:** Update parameters $\boldsymbol{\theta}$ to minimize a loss function

The output of the model is:

$$f(\mathbf{x}; \boldsymbol{\theta}) = \langle \phi(\mathbf{x}) | U^\dagger(\boldsymbol{\theta}) \, \hat{O} \, U(\boldsymbol{\theta}) | \phi(\mathbf{x}) \rangle$$

where $\hat{O}$ is a measurement observable (e.g., $Z$ on the first qubit).

### 1.3 Potential Quantum Advantages

- **Expressibility:** Quantum circuits can represent functions that are hard to compute classically
- **Kernel methods:** Quantum feature maps can define kernels that are classically intractable
- **Data structure:** For data with quantum-like correlations, quantum models may be natural

## 2. Data Encoding Strategies

### 2.1 Basis Encoding

Maps binary strings to computational basis states:

$$\mathbf{x} = (x_1, x_2, \ldots, x_n) \in \{0,1\}^n \quad \mapsto \quad |x_1 x_2 \cdots x_n\rangle$$

- **Qubits needed:** $n$ qubits for $n$-bit data
- **Pro:** Simple, direct
- **Con:** Only encodes binary data, no superposition benefit

### 2.2 Amplitude Encoding

Encodes a normalized vector $\mathbf{x} \in \mathbb{R}^{2^n}$ as amplitudes:

$$\mathbf{x} = (x_0, x_1, \ldots, x_{2^n-1}) \quad \mapsto \quad |\psi_x\rangle = \sum_{i=0}^{2^n - 1} x_i |i\rangle$$

where $\|\mathbf{x}\|_2 = 1$.

- **Qubits needed:** $n = \lceil \log_2 d \rceil$ for $d$-dimensional data (exponentially efficient!)
- **Pro:** Exponential compression of data
- **Con:** State preparation circuit can be $O(2^n)$ deep in general

### 2.3 Angle Encoding (Rotation Encoding)

The most commonly used encoding in variational QML. Each feature is encoded as a rotation angle:

$$\mathbf{x} = (x_1, \ldots, x_n) \quad \mapsto \quad |\phi(\mathbf{x})\rangle = \bigotimes_{i=1}^n R_Y(x_i)|0\rangle$$

where $R_Y(\theta) = e^{-i\theta Y/2} = \begin{pmatrix} \cos(\theta/2) & -\sin(\theta/2) \\ \sin(\theta/2) & \cos(\theta/2) \end{pmatrix}$.

- **Qubits needed:** $n$ qubits for $n$ features
- **Pro:** Natural, shallow circuits, works for continuous data
- **Con:** Linear qubit cost

### 2.4 IQP (Instantaneous Quantum Polynomial) Encoding

A more expressive encoding using entanglement:

$$|\phi(\mathbf{x})\rangle = U_{\text{ent}} \cdot H^{\otimes n} \cdot U_Z(\mathbf{x}) \cdot H^{\otimes n} |0\rangle^{\otimes n}$$

where $U_Z(\mathbf{x}) = \exp\left(i \sum_i x_i Z_i + i \sum_{i<j} x_i x_j Z_i Z_j\right)$.

In [ ]:
# pip install pennylane matplotlib numpy scikit-learn

import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt

# Demonstrate different encoding strategies

# 1. Angle Encoding
n_qubits = 4
dev = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev)
def angle_encoding(x):
    """Encode data using rotation gates."""
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)
    return qml.state()

# Test with sample data
x_sample = np.array([0.5, 1.2, 0.8, 2.1])
state = angle_encoding(x_sample)
print("Angle Encoding")
print(f"Input: {x_sample}")
print(f"State vector (first 8 amplitudes): {np.round(state[:8], 4)}")
print(f"State norm: {np.sum(np.abs(state)**2):.6f}")

In [ ]:
# 2. Amplitude Encoding
dev_amp = qml.device('default.qubit', wires=2)  # 2 qubits for 4-dim data

@qml.qnode(dev_amp)
def amplitude_encoding(x):
    """Encode normalized data as amplitudes."""
    qml.AmplitudeEmbedding(features=x, wires=range(2), normalize=True)
    return qml.state()

x_amp = np.array([1.0, 2.0, 3.0, 4.0])
state_amp = amplitude_encoding(x_amp)
print("Amplitude Encoding")
print(f"Input: {x_amp}")
print(f"Normalized: {x_amp / np.linalg.norm(x_amp)}")
print(f"State: {np.round(state_amp, 4)}")
print(f"Only {2} qubits needed for {len(x_amp)}-dimensional data!")

In [ ]:
# 3. IQP / ZZ Feature Map Encoding
dev_iqp = qml.device('default.qubit', wires=3)

@qml.qnode(dev_iqp)
def iqp_encoding(x):
    """IQP-style encoding with ZZ interactions."""
    # First layer: H + RZ
    for i in range(3):
        qml.Hadamard(wires=i)
        qml.RZ(x[i], wires=i)
    
    # ZZ interactions
    for i in range(3):
        for j in range(i+1, 3):
            qml.CNOT(wires=[i, j])
            qml.RZ(x[i] * x[j], wires=j)
            qml.CNOT(wires=[i, j])
    
    # Second layer
    for i in range(3):
        qml.Hadamard(wires=i)
        qml.RZ(x[i], wires=i)
    
    return qml.state()

x_iqp = np.array([0.5, 1.0, 1.5])
state_iqp = iqp_encoding(x_iqp)
print("IQP Encoding")
print(f"Input: {x_iqp}")
print(f"State (first 8 amplitudes): {np.round(state_iqp, 4)}")
print(f"\nThe IQP encoding creates correlations between features via ZZ gates.")

## 3. Parameterized Quantum Circuits (PQCs)

### 3.1 Structure

A **Parameterized Quantum Circuit** (also called a **variational ansatz**) is a quantum circuit with tunable parameters:

$$U(\boldsymbol{\theta}) = \prod_{l=1}^{L} W_l(\boldsymbol{\theta}_l) V_l$$

where:
- $W_l(\boldsymbol{\theta}_l)$: parameterized rotation gates (e.g., $R_Y(\theta), R_Z(\theta)$)
- $V_l$: fixed entangling gates (e.g., CNOTs)
- $L$: number of layers (circuit depth)

### 3.2 Common Ansatz Designs

**Hardware-efficient ansatz:** Alternating rotation and entangling layers:

$$U(\boldsymbol{\theta}) = \prod_{l=1}^{L} \left[ \text{CNOT\_layer} \cdot \bigotimes_{i=1}^n R_Y(\theta_{l,i}^Y) R_Z(\theta_{l,i}^Z) \right]$$

**Strongly entangling layers:** Each qubit gets 3 rotation parameters and is entangled with two other qubits.

### 3.3 Expressibility

The **expressibility** of a PQC measures how well it covers the space of unitary operators. It is quantified by the distance between the distribution of fidelities generated by the PQC and the Haar-random distribution:

$$\text{Expr} = D_{KL}\left(\hat{P}_{\text{PQC}}(F) \| P_{\text{Haar}}(F)\right)$$

where $F = |\langle 0 | U^\dagger(\boldsymbol{\theta}_1) U(\boldsymbol{\theta}_2) | 0 \rangle|^2$.

In [ ]:
# Build and visualize different PQC architectures

n_qubits = 4
n_layers = 2
dev_pqc = qml.device('default.qubit', wires=n_qubits)

def hardware_efficient_ansatz(params, wires):
    """Hardware-efficient ansatz with RY-RZ rotations and CNOT entanglement."""
    n_wires = len(wires)
    n_layers = params.shape[0]
    
    for l in range(n_layers):
        # Rotation layer
        for i in range(n_wires):
            qml.RY(params[l, i, 0], wires=wires[i])
            qml.RZ(params[l, i, 1], wires=wires[i])
        
        # Entangling layer (linear chain)
        for i in range(n_wires - 1):
            qml.CNOT(wires=[wires[i], wires[i + 1]])

@qml.qnode(dev_pqc)
def pqc_circuit(params, x):
    """Full QML circuit: encoding + PQC + measurement."""
    # Data encoding
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)
    
    # PQC
    hardware_efficient_ansatz(params, wires=range(n_qubits))
    
    return qml.expval(qml.PauliZ(0))

# Random parameters
np.random.seed(42)
params = np.random.uniform(0, 2*np.pi, size=(n_layers, n_qubits, 2))
x_test = np.array([0.5, 1.0, 1.5, 2.0])

result = pqc_circuit(params, x_test)
print(f"PQC output for x={x_test}: <Z_0> = {result:.4f}")
print(f"\nTotal parameters: {params.size} ({n_layers} layers x {n_qubits} qubits x 2 rotations)")

# Draw the circuit
print("\nCircuit diagram:")
print(qml.draw(pqc_circuit)(params, x_test))

## 4. Quantum Feature Maps and Kernels

### 4.1 Feature Map

A **quantum feature map** $\phi$ maps classical data to quantum states:

$$\phi: \mathbb{R}^d \to \mathcal{H}_{2^n}, \quad \mathbf{x} \mapsto |\phi(\mathbf{x})\rangle = U_{\phi}(\mathbf{x})|0\rangle^{\otimes n}$$

### 4.2 Quantum Kernel

The **quantum kernel** is the inner product of feature-mapped states:

$$\kappa(\mathbf{x}, \mathbf{x}') = |\langle \phi(\mathbf{x}) | \phi(\mathbf{x}') \rangle|^2$$

This is the probability of measuring $|0\rangle^{\otimes n}$ after applying $U_\phi^\dagger(\mathbf{x}) U_\phi(\mathbf{x}')$.

### 4.3 Connection to Reproducing Kernel Hilbert Spaces

The quantum kernel defines a **reproducing kernel Hilbert space (RKHS)**. The QML model becomes:

$$f(\mathbf{x}) = \sum_{i=1}^{N} \alpha_i \kappa(\mathbf{x}_i, \mathbf{x})$$

This is the **kernel trick**: we never need to explicitly compute the (exponentially large) feature vector!

### 4.4 Representational Power

The function computed by a quantum model can be expressed as a **Fourier series**:

$$f(\mathbf{x}) = \sum_{\omega \in \Omega} c_\omega e^{i \omega \cdot \mathbf{x}}$$

where $\Omega$ is the set of accessible frequencies, determined by the eigenvalues of the data-encoding gates. The **data re-uploading** technique (encoding data in multiple layers) enriches $\Omega$.

In [ ]:
# Compute and visualize quantum kernels

n_q = 2
dev_kernel = qml.device('default.qubit', wires=n_q)

@qml.qnode(dev_kernel)
def kernel_circuit(x1, x2):
    """Compute quantum kernel κ(x1, x2) = |<φ(x1)|φ(x2)>|²."""
    # Prepare |φ(x2)>
    for i in range(n_q):
        qml.RY(x2[i], wires=i)
    for i in range(n_q - 1):
        qml.CNOT(wires=[i, i + 1])
    for i in range(n_q):
        qml.RZ(x2[i], wires=i)
    
    # Apply U†(x1) = adjoint of encoding
    for i in range(n_q - 1, -1, -1):
        qml.RZ(-x1[i], wires=i)
    for i in range(n_q - 2, -1, -1):
        qml.CNOT(wires=[i, i + 1])
    for i in range(n_q - 1, -1, -1):
        qml.RY(-x1[i], wires=i)
    
    return qml.probs(wires=range(n_q))

def quantum_kernel(x1, x2):
    """Compute the quantum kernel value."""
    probs = kernel_circuit(x1, x2)
    return probs[0]  # Probability of |00...0>

# Generate sample data
np.random.seed(42)
n_samples = 30
X_data = np.random.uniform(0, 2 * np.pi, size=(n_samples, n_q))

# Compute kernel matrix
K = np.zeros((n_samples, n_samples))
for i in range(n_samples):
    for j in range(i, n_samples):
        K[i, j] = quantum_kernel(X_data[i], X_data[j])
        K[j, i] = K[i, j]

# Visualize kernel matrix
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(K, cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Kernel value κ(x_i, x_j)')
ax.set_xlabel('Sample index', fontsize=12)
ax.set_ylabel('Sample index', fontsize=12)
ax.set_title('Quantum Kernel Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Kernel matrix shape: {K.shape}")
print(f"Diagonal values (should be 1): {K[0,0]:.4f}, {K[1,1]:.4f}, {K[2,2]:.4f}")
print(f"Matrix is positive semidefinite: {np.all(np.linalg.eigvalsh(K) >= -1e-10)}")

## 5. The Barren Plateaus Problem

### 5.1 What Are Barren Plateaus?

**Barren plateaus** are regions in the parameter landscape where gradients vanish exponentially with system size. Formally, for a cost function $C(\boldsymbol{\theta})$:

$$\text{Var}_{\boldsymbol{\theta}}\left[\frac{\partial C}{\partial \theta_k}\right] \leq F(n)$$

where $F(n)$ decreases exponentially in $n$ (number of qubits).

### 5.2 When Do Barren Plateaus Occur?

Barren plateaus arise when:

1. **Deep random circuits:** If the PQC forms a 2-design (sufficiently random), then:
   $$\text{Var}\left[\frac{\partial C}{\partial \theta_k}\right] = O\left(\frac{1}{2^n}\right)$$

2. **Global cost functions:** Measuring global observables (e.g., all qubits) leads to exponentially vanishing gradients

3. **Excessive entanglement:** Highly entangled states have exponentially small local gradients

4. **Hardware noise:** Noise effectively depolarizes the state, flattening the landscape

### 5.3 Mitigation Strategies

- **Local cost functions:** Use observables acting on few qubits
- **Shallow circuits:** Keep depth $O(\log n)$ or $O(1)$
- **Structured ansatze:** Use problem-specific circuit designs
- **Layer-wise training:** Train one layer at a time
- **Parameter initialization:** Start near identity or use pre-trained parameters
- **Classical pre-training:** Initialize with classically computed parameters

In [ ]:
# Demonstrate barren plateaus: gradient variance vs number of qubits

def compute_gradient_variance(n_qubits, n_layers, n_samples=200):
    """Estimate variance of gradient for random PQC parameters."""
    dev = qml.device('default.qubit', wires=n_qubits)
    
    @qml.qnode(dev, diff_method='parameter-shift')
    def circuit(params):
        for l in range(n_layers):
            for i in range(n_qubits):
                qml.RY(params[l, i], wires=i)
            for i in range(n_qubits - 1):
                qml.CNOT(wires=[i, i + 1])
        return qml.expval(qml.PauliZ(0))
    
    gradients = []
    for _ in range(n_samples):
        params = np.random.uniform(0, 2 * np.pi, size=(n_layers, n_qubits))
        grad = qml.grad(circuit)(params)
        gradients.append(grad[0, 0])  # Gradient w.r.t. first parameter
    
    return np.var(gradients)

# Compute for different qubit counts
qubit_range = [2, 3, 4, 5, 6, 7, 8]
n_layers_bp = 3
variances = []

print("Computing gradient variances (this may take a minute)...")
for nq in qubit_range:
    var = compute_gradient_variance(nq, n_layers_bp, n_samples=150)
    variances.append(var)
    print(f"  n_qubits={nq}: Var[∂C/∂θ] = {var:.6f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(qubit_range, variances, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Qubits', fontsize=12)
ax1.set_ylabel('Var[∂C/∂θ]', fontsize=12)
ax1.set_title('Gradient Variance vs System Size', fontsize=13)
ax1.grid(True, alpha=0.3)

# Log scale to check exponential decay
ax2.semilogy(qubit_range, variances, 'ro-', linewidth=2, markersize=8)
# Fit exponential
log_var = np.log(variances)
fit = np.polyfit(qubit_range, log_var, 1)
ax2.semilogy(qubit_range, np.exp(np.polyval(fit, qubit_range)), 'k--', 
             label=f'Fit: exp({fit[0]:.2f}n)', linewidth=1.5)
ax2.set_xlabel('Number of Qubits', fontsize=12)
ax2.set_ylabel('Var[∂C/∂θ] (log scale)', fontsize=12)
ax2.set_title('Exponential Decay of Gradients', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nExponential fit: Var ~ exp({fit[0]:.3f} * n)")
print(f"This confirms barren plateaus: gradients vanish exponentially!")

## 6. Expressibility and Entangling Capability

### 6.1 Expressibility

Expressibility quantifies how uniformly a PQC can explore the Hilbert space. For $n$ qubits, the Haar-random fidelity distribution is:

$$P_{\text{Haar}}(F) = (2^n - 1)(1 - F)^{2^n - 2}$$

A PQC with high expressibility has its fidelity distribution close to this.

### 6.2 Entangling Capability

The **entangling capability** measures how much entanglement the PQC can generate:

$$\text{Ent} = \mathbb{E}_{\boldsymbol{\theta}} \left[ Q(U(\boldsymbol{\theta})|0\rangle^{\otimes n}) \right]$$

where $Q$ is the Meyer-Wallach entanglement measure:

$$Q(|\psi\rangle) = \frac{2}{n} \sum_{k=1}^{n} \left(1 - \text{Tr}(\rho_k^2)\right)$$

with $\rho_k$ being the reduced density matrix of qubit $k$.

### 6.3 The Expressibility-Trainability Tradeoff

There is a fundamental tradeoff:
- **High expressibility** $\to$ more barren plateaus $\to$ harder to train
- **Low expressibility** $\to$ may not capture the target function

The art of QML circuit design is finding the right balance.

In [ ]:
# Measure expressibility of different ansatze

def compute_fidelities(circuit_fn, n_qubits, n_samples=500):
    """Compute fidelity distribution F = |<0|U†(θ1)U(θ2)|0>|²."""
    dev = qml.device('default.qubit', wires=n_qubits)
    
    @qml.qnode(dev)
    def fidelity_circuit(params1, params2):
        circuit_fn(params2)
        qml.adjoint(circuit_fn)(params1)
        return qml.probs(wires=range(n_qubits))
    
    fidelities = []
    param_shape = circuit_fn.param_shape if hasattr(circuit_fn, 'param_shape') else None
    
    for _ in range(n_samples):
        p1 = np.random.uniform(0, 2*np.pi, size=(2, n_qubits))
        p2 = np.random.uniform(0, 2*np.pi, size=(2, n_qubits))
        probs = fidelity_circuit(p1, p2)
        fidelities.append(probs[0])  # |<0...0|...>|²
    
    return np.array(fidelities)

nq = 3

# Ansatz 1: Only rotations (no entanglement)
def ansatz_no_ent(params):
    for l in range(2):
        for i in range(nq):
            qml.RY(params[l, i], wires=i)

# Ansatz 2: Rotations + linear CNOT
def ansatz_linear(params):
    for l in range(2):
        for i in range(nq):
            qml.RY(params[l, i], wires=i)
        for i in range(nq - 1):
            qml.CNOT(wires=[i, i + 1])

# Ansatz 3: Rotations + all-to-all CNOT
def ansatz_full(params):
    for l in range(2):
        for i in range(nq):
            qml.RY(params[l, i], wires=i)
        for i in range(nq):
            for j in range(i + 1, nq):
                qml.CNOT(wires=[i, j])

# Compute fidelity distributions
print("Computing fidelity distributions...")
F_no_ent = compute_fidelities(ansatz_no_ent, nq, n_samples=400)
F_linear = compute_fidelities(ansatz_linear, nq, n_samples=400)
F_full = compute_fidelities(ansatz_full, nq, n_samples=400)

# Haar random reference
F_haar = np.linspace(0, 1, 200)
P_haar = (2**nq - 1) * (1 - F_haar)**(2**nq - 2)

fig, ax = plt.subplots(figsize=(10, 6))
bins = np.linspace(0, 1, 40)
ax.hist(F_no_ent, bins=bins, density=True, alpha=0.5, label='No entanglement')
ax.hist(F_linear, bins=bins, density=True, alpha=0.5, label='Linear CNOT')
ax.hist(F_full, bins=bins, density=True, alpha=0.5, label='Full CNOT')
ax.plot(F_haar, P_haar, 'k--', linewidth=2, label='Haar random')

ax.set_xlabel('Fidelity F', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title(f'Expressibility of Different Ansatze ({nq} qubits)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Building a Quantum Classifier

### 7.1 The Classification Task

We will build a quantum classifier for a simple 2D dataset. The model:

$$\hat{y}(\mathbf{x}) = \text{sign}\left(\langle \phi(\mathbf{x}) | U^\dagger(\boldsymbol{\theta}) Z_0 U(\boldsymbol{\theta}) | \phi(\mathbf{x}) \rangle\right)$$

### 7.2 Loss Function

We use the **mean squared error**:

$$\mathcal{L}(\boldsymbol{\theta}) = \frac{1}{N} \sum_{i=1}^N \left(y_i - f(\mathbf{x}_i; \boldsymbol{\theta})\right)^2$$

### 7.3 Gradient Computation

Gradients are computed using the **parameter-shift rule**:

$$\frac{\partial f}{\partial \theta_k} = \frac{f(\theta_k + \pi/2) - f(\theta_k - \pi/2)}{2}$$

This is exact (not an approximation!) and requires only two circuit evaluations per parameter.

In [ ]:
# Generate a classification dataset
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Create moon dataset
X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
y = 2 * y - 1  # Convert to {-1, +1}

# Scale features to [0, pi]
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

# Visualize the dataset
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='RdBu', 
                     edgecolors='black', s=50, alpha=0.8)
ax.set_xlabel('Feature 1', fontsize=12)
ax.set_ylabel('Feature 2', fontsize=12)
ax.set_title('Moon Dataset for Quantum Classification', fontsize=14)
plt.colorbar(scatter, label='Class')
plt.tight_layout()
plt.show()

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Build quantum classifier with PennyLane

n_qubits_cls = 2
n_layers_cls = 4
dev_cls = qml.device('default.qubit', wires=n_qubits_cls)

@qml.qnode(dev_cls, diff_method='parameter-shift')
def quantum_classifier(weights, x):
    """Quantum classifier with data re-uploading."""
    for l in range(n_layers_cls):
        # Data encoding (re-uploaded each layer)
        for i in range(n_qubits_cls):
            qml.RX(x[i], wires=i)
        
        # Variational layer
        for i in range(n_qubits_cls):
            qml.RY(weights[l, i, 0], wires=i)
            qml.RZ(weights[l, i, 1], wires=i)
        
        # Entanglement
        qml.CNOT(wires=[0, 1])
    
    return qml.expval(qml.PauliZ(0))

def cost_function(weights, X, y):
    """Mean squared error loss."""
    predictions = np.array([quantum_classifier(weights, x) for x in X])
    return np.mean((predictions - y) ** 2)

def accuracy(weights, X, y):
    """Classification accuracy."""
    predictions = np.array([quantum_classifier(weights, x) for x in X])
    predicted_labels = np.sign(predictions)
    return np.mean(predicted_labels == y)

# Initialize weights
np.random.seed(42)
weights = np.random.uniform(-np.pi, np.pi, size=(n_layers_cls, n_qubits_cls, 2))

print(f"Number of parameters: {weights.size}")
print(f"Initial cost: {cost_function(weights, X_train[:20], y_train[:20]):.4f}")
print(f"Initial accuracy: {accuracy(weights, X_test[:20], y_test[:20]):.2%}")

In [ ]:
# Train the quantum classifier

# Use a subset for faster training
n_train = 60
n_test = 30
X_tr = X_train[:n_train]
y_tr = y_train[:n_train]
X_te = X_test[:n_test]
y_te = y_test[:n_test]

# Gradient descent with PennyLane
opt = qml.GradientDescentOptimizer(stepsize=0.3)

n_epochs = 30
batch_size = 15
costs = []
train_accs = []
test_accs = []

print("Training quantum classifier...")
print(f"{'Epoch':>6} {'Cost':>10} {'Train Acc':>12} {'Test Acc':>12}")
print("-" * 42)

for epoch in range(n_epochs):
    # Mini-batch SGD
    batch_idx = np.random.choice(n_train, size=batch_size, replace=False)
    X_batch = X_tr[batch_idx]
    y_batch = y_tr[batch_idx]
    
    weights = opt.step(lambda w: cost_function(w, X_batch, y_batch), weights)
    
    # Evaluate every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        c = cost_function(weights, X_tr, y_tr)
        tr_acc = accuracy(weights, X_tr, y_tr)
        te_acc = accuracy(weights, X_te, y_te)
        costs.append(c)
        train_accs.append(tr_acc)
        test_accs.append(te_acc)
        print(f"{epoch+1:6d} {c:10.4f} {tr_acc:12.2%} {te_acc:12.2%}")

print("\nTraining complete!")

In [ ]:
# Visualize decision boundary

# Create a mesh grid
x_min, x_max = 0, np.pi
y_min, y_max = 0, np.pi
h = 0.05
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Predict on grid
print("Computing decision boundary...")
Z_pred = np.array([quantum_classifier(weights, x) for x in grid_points])
Z_pred = Z_pred.reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Decision boundary
ax1.contourf(xx, yy, Z_pred, levels=20, cmap='RdBu', alpha=0.6)
ax1.contour(xx, yy, Z_pred, levels=[0], colors='black', linewidths=2)
ax1.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap='RdBu', edgecolors='black', s=60)
ax1.set_xlabel('Feature 1', fontsize=12)
ax1.set_ylabel('Feature 2', fontsize=12)
ax1.set_title('Quantum Classifier Decision Boundary', fontsize=13)

# Training curves
epochs_plot = list(range(1, len(costs) + 1))
ax2.plot(epochs_plot, train_accs, 'bo-', label='Train accuracy', linewidth=2)
ax2.plot(epochs_plot, test_accs, 'rs-', label='Test accuracy', linewidth=2)
ax2.set_xlabel('Evaluation Step', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Training Progress', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 8. Data Re-uploading and Expressiveness

### 8.1 The Idea

In our classifier, we **re-upload** the input data in every layer. This is crucial because:

Without re-uploading, a single-qubit model computes:
$$f(x) = A + B\cos(x) + C\sin(x)$$

With $L$ layers of re-uploading, it can compute:
$$f(x) = \sum_{k=0}^{L} \left(a_k \cos(kx) + b_k \sin(kx)\right)$$

a **Fourier series** of order $L$! This is a universal function approximator as $L \to \infty$.

### 8.2 Theorem (Perez-Salinas et al., 2020)

> A single qubit with data re-uploading is a universal quantum classifier. With $L$ layers, it can approximate any continuous function on a compact domain to accuracy $\epsilon$ with $L = O(1/\epsilon)$.

In [ ]:
# Demonstrate how data re-uploading increases model expressiveness

def single_qubit_model(x, weights, n_layers):
    """Single-qubit model with data re-uploading."""
    dev_1q = qml.device('default.qubit', wires=1)
    
    @qml.qnode(dev_1q)
    def circuit(x, w):
        for l in range(n_layers):
            qml.RX(x, wires=0)
            qml.RY(w[l, 0], wires=0)
            qml.RZ(w[l, 1], wires=0)
        return qml.expval(qml.PauliZ(0))
    
    return circuit(x, weights)

# Show how the function space grows with layers
x_range = np.linspace(0, 2*np.pi, 200)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for idx, n_lay in enumerate([1, 2, 3, 5, 8, 12]):
    ax = axes[idx // 3][idx % 3]
    
    # Plot several random function instances
    for trial in range(8):
        np.random.seed(trial + idx * 10)
        w = np.random.uniform(-np.pi, np.pi, size=(n_lay, 2))
        y_vals = [single_qubit_model(x, w, n_lay) for x in x_range]
        ax.plot(x_range, y_vals, alpha=0.6, linewidth=1)
    
    ax.set_title(f'L = {n_lay} layers (Fourier order {n_lay})', fontsize=11)
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.set_ylim(-1.1, 1.1)
    ax.grid(True, alpha=0.3)

plt.suptitle('Single-Qubit Model: Expressiveness vs Number of Re-uploading Layers', fontsize=14)
plt.tight_layout()
plt.show()

print("More layers = higher Fourier frequencies = more expressive functions")

## 9. Summary and Key Takeaways

### What We Learned

| Concept | Key Insight |
|---------|------------|
| QML pipeline | Encode $\to$ Process $\to$ Measure $\to$ Optimize |
| Basis encoding | Binary data $\to$ computational basis ($n$ qubits) |
| Amplitude encoding | $d$-dim vector $\to$ $\log_2 d$ qubits (exponential compression) |
| Angle encoding | Features as rotation angles ($n$ qubits for $n$ features) |
| PQCs | $U(\theta)$ = parameterized rotations + entangling gates |
| Quantum kernel | $\kappa(x,x') = |\langle\phi(x)|\phi(x')\rangle|^2$ |
| Barren plateaus | Var[$\partial C/\partial\theta$] $\sim O(1/2^n)$ for deep random circuits |
| Parameter-shift rule | Exact gradients: $(f(\theta+\pi/2) - f(\theta-\pi/2))/2$ |
| Data re-uploading | $L$ layers = Fourier series of order $L$ = universal approximator |

### Looking Ahead

In the next notebook, we will dive deeper into **Quantum Kernel Methods**, implementing quantum-enhanced SVMs and exploring conditions for quantum advantage.

---

*Notebook 14 of the Quantum Computing from Ground Up series.*